# FortiOS 8 Lab - Remote Service Client
## Connect to deployed lab service (Render, Fly.io, Docker, or Custom)

Use this notebook to control a remotely deployed lab service.

---

## STEP 1: Download CLI

In [ ]:
import os
import subprocess
import urllib.request
from pathlib import Path

print("🚀 Setting up remote client...\n")

LAB_DIR = Path('/content/fortios_client')
LAB_DIR.mkdir(exist_ok=True)
os.chdir(LAB_DIR)

# Install dependencies
print("[*] Installing dependencies...")
subprocess.run(['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'requests'],
               check=False)

# Download CLI
print("[*] Downloading CLI tool...")
base = "https://raw.githubusercontent.com/netanelcyber/HAMIVTZAR/claude/cve-2026-24858-fortios-sso-p1lrtu/cve-research/CVE-SUSPECTED-FORTIOS8-SSO/"
urllib.request.urlretrieve(base + 'kvm_lab_cli.py', 'kvm_lab_cli.py')
os.chmod('kvm_lab_cli.py', 0o755)

print("\n✅ Setup complete")

## STEP 2: Configure Remote Service

In [ ]:
# Remote service configuration
# Change these to match your deployed service

# Option A: Render.com
# SERVICE_URL = "https://your-service-name.onrender.com"

# Option B: Fly.io
# SERVICE_URL = "https://your-app-name.fly.dev"

# Option C: Docker (Local)
# SERVICE_URL = "http://localhost:5000"

# Option D: Custom Server
SERVICE_URL = "http://localhost:5000"  # ← Edit this

# API Key (auto-generated on service startup)
# Retrieve from service logs or endpoint
API_KEY = None  # Will be auto-retrieved

print("🔧 Remote Service Configuration:")
print(f"   Service URL: {SERVICE_URL}")
print(f"   API Key: {'Auto-retrieve' if not API_KEY else 'Set'}")
print(f"\n📝 Edit SERVICE_URL to your deployed service")

## STEP 3: Retrieve Auto-Generated API Key

In [ ]:
import requests

print("🔐 Retrieving auto-generated API key...\n")

try:
    response = requests.get(f'{SERVICE_URL}/api/key', timeout=10)
    API_KEY = response.json().get('api_key')
    
    if API_KEY:
        print(f"✅ API Key retrieved")
        print(f"\n   Key: {API_KEY}")
        print(f"\n   Use this key for authenticated requests:")
        print(f"   curl -H 'X-API-Key: {API_KEY}' {SERVICE_URL}/api/state")
    else:
        print("❌ No API key found")
except Exception as e:
    print(f"❌ Error: {e}")
    print(f"\n   Make sure the service is running at: {SERVICE_URL}")

## STEP 4: Test Service Connection

In [ ]:
import requests
import json

print("🔍 Testing service connection...\n")

try:
    headers = {'X-API-Key': API_KEY} if API_KEY else {}
    response = requests.get(f'{SERVICE_URL}/health', headers=headers, timeout=10)
    
    if response.status_code == 200:
        data = response.json()
        print("✅ Service is running\n")
        print("📊 Health Status:")
        print(json.dumps(data, indent=2))
    else:
        print(f"❌ Service returned: {response.status_code}")
except Exception as e:
    print(f"❌ Connection failed: {e}")

## STEP 5: Lab Control - Health Check

In [ ]:
# Use CLI with remote service
import os
os.environ['KVM_LAB_URL'] = SERVICE_URL
os.environ['API_KEY'] = API_KEY or ''

!python3 kvm_lab_cli.py --url "$KVM_LAB_URL" health

## STEP 6: Lab Control - Get Configuration

In [ ]:
!python3 kvm_lab_cli.py --url "$KVM_LAB_URL" config

## STEP 7: Lab Control - Connect KVM

In [ ]:
!python3 kvm_lab_cli.py --url "$KVM_LAB_URL" connect

## STEP 8: Lab Control - Setup

In [ ]:
!python3 kvm_lab_cli.py --url "$KVM_LAB_URL" setup

## STEP 9: Lab Control - Tunnel

In [ ]:
!python3 kvm_lab_cli.py --url "$KVM_LAB_URL" tunnel-start

## STEP 10: Lab Control - Exploitation

In [ ]:
!python3 kvm_lab_cli.py --url "$KVM_LAB_URL" exploit --target-ip 127.0.0.1 --target-port 12443

## STEP 11: Lab Control - Reconnaissance

In [ ]:
!python3 kvm_lab_cli.py --url "$KVM_LAB_URL" recon --target-ip 127.0.0.1 --target-port 12443

## STEP 12: View Real-Time Logs

In [ ]:
print("📋 Lab Activity Logs\n")

!python3 kvm_lab_cli.py --url "$KVM_LAB_URL" logs --limit 50

In [ ]:
# Advanced log analysis
import requests
import json

headers = {'X-API-Key': API_KEY} if API_KEY else {}
response = requests.get(f'{SERVICE_URL}/api/logs?limit=100', headers=headers)
logs = response.json().get('logs', [])

print(f"\n📊 Log Analysis (showing {len(logs)} of {response.json().get('total', 0)} entries)\n")

# Group by level
by_level = {}
for log in logs:
    level = log.get('level', 'UNKNOWN')
    if level not in by_level:
        by_level[level] = []
    by_level[level].append(log)

# Display summary
for level in ['ERROR', 'WARN', 'INFO', 'DEBUG']:
    if level in by_level:
        count = len(by_level[level])
        icon = {'ERROR': '✗', 'WARN': '⚠', 'INFO': '✓', 'DEBUG': '→'}.get(level, '•')
        print(f"{icon} {level:6} {count:3} entries")
        if level == 'ERROR' and count > 0:
            for err in by_level[level][-3:]:
                print(f"        - {err.get('message')}")

## STEP 13: Lab Control - Status

In [ ]:
!python3 kvm_lab_cli.py --url "$KVM_LAB_URL" status

## STEP 14: Lab Control - Cleanup

In [ ]:
!python3 kvm_lab_cli.py --url "$KVM_LAB_URL" cleanup

## Direct Python API Access

In [ ]:
import requests
import json

headers = {'X-API-Key': API_KEY} if API_KEY else {}

print("🔗 Direct API Access Examples\n")

# Example 1: Get state
print("[1] Get Lab State:")
r = requests.get(f'{SERVICE_URL}/api/state', headers=headers)
print(json.dumps(r.json(), indent=2))

# Example 2: Get logs
print("\n[2] Get Recent Logs:")
r = requests.get(f'{SERVICE_URL}/api/logs?limit=5', headers=headers)
logs = r.json().get('logs', [])
for log in logs:
    print(f"  {log.get('timestamp')[:19]} [{log.get('level'):6}] {log.get('message')}")